# 03 — Run Exp 1: Non-bridge edge removal (δ_min-stratified)

Runs `ANALYSIS/perturbation/pe_stability_exp1.py` on each of the 5 PE methods using the trained checkpoints you already have on Drive.

**Prerequisites**
1. `MyDrive/datasets/pcqm4m-v2/processed/` contains the OGB-processed PCQM4Mv2 `.pt` file (run `00_setup.ipynb` once).
2. Baseline checkpoints live under **`MyDrive/results_pcqm4m_subset/stability_baselines/`** with short dir names: `lappe/`, `snmlp/` (SignNet-MLP), `snds/` (SignNet-DeepSets), each with `seed0/`, `seed1/`, … containing `config.yaml` + `*.ckpt` under the run subfolder. L-HKS / fix-L-HKS use `mlp_ablation/` under the same Drive folder (see `METHOD_RUN_DIRS` in section 1).

**Drive account:** **`znidar.mark@gmail.com`** (authorize in the Drive popup).

**Time budget on A100, 5000 graphs × 3 k_remove × 20 perturbations per checkpoint:** ≈40–60 min / seed. With 7 checkpoints total that's ~5–7 h. Use `N_GRAPHS = 500` first to sanity-check end-to-end before the full sweep.

## 1. Configuration — edit paths and seeds

`METHOD_RUN_DIRS` lists per-seed run directories (each tree contains `config.yaml` + a `.ckpt`). Paths are repo-relative; the `results_pcqm4m_subset` symlink in step 4 points at `MyDrive/results_pcqm4m_subset/` on Drive.

In [ ]:
# =========================================================
# EDIT: which checkpoints to evaluate (repo-relative; Drive via symlinks)
# =========================================================
METHOD_RUN_DIRS = {
    "LapPE":            ["results_pcqm4m_subset/stability_baselines/lappe/seed1"],
    "SignNet-MLP":      ["results_pcqm4m_subset/stability_baselines/snmlp/seed1"],
    "SignNet-DeepSets": ["results_pcqm4m_subset/stability_baselines/snds/seed1"],
    "L-HKS":            ["results_pcqm4m_subset/mlp_ablation/mlp3/seed2"],
    "fix-L-HKS":        ["results_pcqm4m_subset/mlp_ablation/mlp3_fixed/seed5"],
}

# EDIT: smaller number for a fast sanity check (e.g. 500); 5000 for final figure.
N_GRAPHS   = 5000
BATCH_SIZE = 64

GITHUB_REPO     = "https://github.com/mark-znidar/gdl__2.git"
DRIVE_WORKSPACE = "/content/drive/MyDrive"
# =========================================================

DRIVE_DATASETS        = f"{DRIVE_WORKSPACE}/datasets"
DRIVE_RESULTS         = f"{DRIVE_WORKSPACE}/results_pcqm4m_subset"
DRIVE_EXP_RESULTS     = f"{DRIVE_WORKSPACE}/pe_stability_results/exp1"

import os
os.makedirs(DRIVE_EXP_RESULTS, exist_ok=True)

print("Methods to evaluate:")
for m, dirs in METHOD_RUN_DIRS.items():
    for d in dirs:
        print(f"  {m:18s}  {d}")
print(f"\nN_GRAPHS = {N_GRAPHS}, BATCH_SIZE = {BATCH_SIZE}")
print(f"Exp-1 results will be written to: {DRIVE_EXP_RESULTS}")

## 2. GPU check

A100 / V100 is ideal; T4 works but will roughly double the time.

In [ ]:
!nvidia-smi | head -20

## 3. Mount Drive

In [ ]:
from google.colab import drive
import os, shutil

MOUNTPOINT = "/content/drive"
# Colab errors if MOUNTPOINT already has files (re-run order, failed mount, etc.)
try:
    drive.flush_and_unmount()
except Exception:
    pass
if os.path.islink(MOUNTPOINT):
    os.remove(MOUNTPOINT)
elif os.path.isdir(MOUNTPOINT) and os.listdir(MOUNTPOINT):
    shutil.rmtree(MOUNTPOINT)
os.makedirs(MOUNTPOINT, exist_ok=True)
drive.mount(MOUNTPOINT)

## 4. Clone repo + install deps + symlink Drive folders

`datasets/`, `results_pcqm4m_subset/` (includes `stability_baselines/` + `mlp_ablation/` checkpoints), and the experiment output folder are symlinked into the repo so reads/writes go to Drive and survive Colab VM restarts.

In [ ]:
import os, subprocess
%cd /content
if not os.path.isdir("gdl__2"):
    !git clone {GITHUB_REPO}
else:
    %cd gdl__2
    !git pull --ff-only || true
    %cd /content
%cd gdl__2
!bash run_colab/setup.sh

# Symlink datasets + results (baselines under results_pcqm4m_subset/stability_baselines/) + exp output
!rm -rf results_pcqm4m_subset datasets
!ln -sfn {DRIVE_RESULTS}    results_pcqm4m_subset
!ln -sfn {DRIVE_DATASETS}   datasets
!mkdir -p ANALYSIS/perturbation
!rm -rf ANALYSIS/perturbation/exp1_results
!ln -sfn {DRIVE_EXP_RESULTS} ANALYSIS/perturbation/exp1_results

!ls -la results_pcqm4m_subset datasets ANALYSIS/perturbation/exp1_results

## 5. Sanity checks — dataset + checkpoints exist

Fails fast if the PCQM4Mv2 processed tensor or any of the listed checkpoints/configs are missing.

In [ ]:
from pathlib import Path
import glob

# --- dataset sanity ---
proc = Path(DRIVE_DATASETS) / "pcqm4m-v2" / "processed"
assert proc.is_dir(), f"Missing {proc}. Run 00_setup.ipynb first."
big = [p for p in proc.glob("*.pt") if p.stat().st_size > 500_000_000]
assert big, "No large processed .pt found in MyDrive/datasets/pcqm4m-v2/processed/"
for p in big:
    print(f"OK dataset  {p.name}  {p.stat().st_size / 1e9:.1f} GB")

# --- checkpoint sanity ---
print("\nCheckpoint discovery:")
missing = []
for method, dirs in METHOD_RUN_DIRS.items():
    for rd in dirs:
        cfg = sorted(glob.glob(os.path.join(rd, "**", "config.yaml"), recursive=True))
        ckpts = sorted(
            glob.glob(os.path.join(rd, "**", "*.ckpt"), recursive=True) +
            glob.glob(os.path.join(rd, "**", "*.pt"),   recursive=True))
        ckpts = [p for p in ckpts if "config" not in os.path.basename(p).lower()]
        if cfg and ckpts:
            print(f"  [OK]   {method:18s} {rd}  ->  cfg={cfg[-1]}  ckpt={ckpts[-1]}")
        else:
            print(f"  [MISS] {method:18s} {rd}  cfg={cfg}  ckpt={ckpts}")
            missing.append((method, rd))
if missing:
    raise FileNotFoundError(
        f"{len(missing)} run dir(s) missing config.yaml or a checkpoint. "
        "Upload under MyDrive/results_pcqm4m_subset/ (see METHOD_RUN_DIRS) and rerun."
    )

## 6. Run the experiment

Loops over every `(method, run_dir)` sequentially. Each call writes one JSON
file into `ANALYSIS/perturbation/exp1_results/` (Drive-backed), so it is safe
to interrupt and re-run: already-completed checkpoints are skipped by the
presence of their result JSON.

Subprocess stdout/stderr is captured and printed in full after each run (so tracebacks appear in this cell on failure).

In [ ]:
import subprocess, datetime, shlex
from pathlib import Path

SCRIPT = "ANALYSIS/perturbation/pe_stability_exp1.py"
OUT_DIR = Path("ANALYSIS/perturbation/exp1_results")

for method, dirs in METHOD_RUN_DIRS.items():
    for rd in dirs:
        seed_tag = Path(rd).name
        out_json = OUT_DIR / f"{method}__{seed_tag}.json"
        if out_json.exists():
            print(f"[skip] {method} :: {rd} — result already exists at {out_json}")
            continue
        cmd = ["python", "-u", SCRIPT,
               "--method", method,
               "--run-dirs", rd,
               "--n-graphs", str(N_GRAPHS),
               "--device", "cuda",
               "--no-plot"]
        print(f"\n{'='*72}")
        print(f">>> {method}  {rd}")
        print(f">>> start: {datetime.datetime.now().isoformat(timespec='seconds')}")
        print(f">>> cmd:   {' '.join(shlex.quote(c) for c in cmd)}")
        print(f"{'='*72}", flush=True)
        proc = subprocess.run(cmd, capture_output=True, text=True)
        ret = proc.returncode
        if proc.stdout:
            print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
        if proc.stderr:
            print("--- subprocess stderr ---\n", proc.stderr, sep="", end="")
        status = "OK" if ret == 0 else f"FAILED (exit {ret})"
        print(f">>> done:  {datetime.datetime.now().isoformat(timespec='seconds')}  [{status}]")
        if ret != 0:
            raise RuntimeError(
                f"Exp1 failed for {method} :: {rd} (exit {ret}). "
                "See full subprocess stdout/stderr printed above this line.")
print("\n=== All Exp 1 runs complete ===")

## 7. Aggregate + plot

Section 6 runs each checkpoint with `--no-plot` (no figure until all JSONs exist). This cell aggregates every `*.json` under `exp1_results/`, writes `exp1_main.png` and `exp1_summary.json`.

In [ ]:
!python ANALYSIS/perturbation/pe_stability_exp1.py --plot-only

In [ ]:
from IPython.display import Image, display
import json
from pathlib import Path

out = Path("ANALYSIS/perturbation/exp1_results")
png = out / "exp1_main.png"
if png.exists():
    display(Image(filename=str(png)))
else:
    print(f"Plot not found at {png}")

summary_path = out / "exp1_summary.json"
if summary_path.exists():
    print("\n--- exp1_summary.json ---")
    print(json.dumps(json.load(open(summary_path)), indent=2))

## 8. Verify results on Drive

Everything under `ANALYSIS/perturbation/exp1_results/` lives on Drive via the symlink — safe across VM restarts and accessible from another machine.

In [ ]:
!ls -la {DRIVE_EXP_RESULTS}
!du -sh {DRIVE_EXP_RESULTS}